In [2]:
import duckdb
duckdb.__version__


'1.4.5'

In [40]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("SET s3_region = 'us-west-2';")
# con.sql("""
# SELECT street_extension_a, COUNT(*) AS N
# FROM read_parquet('Data_Helpers_old/val.parquet')
# GROUP BY street_extension_a
# ORDER BY COUNT(*) DESC
# LIMIT 5
# """).df()
con.sql("""
    DESCRIBE SELECT * FROM read_parquet('s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/part-00000-417d568b-a25f-57b3-9e15-9876068be5d6-c000.zstd.parquet')
""")

# con.sql("""
# SELECT id, names.primary, addresses, sources
# FROM read_parquet('s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/part-00000-417d568b-a25f-57b3-9e15-9876068be5d6-c000.zstd.parquet')
# LIMIT 5
# """)

# con.sql("""
# SELECT addresses
# FROM read_parquet('s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/*.zstd.parquet', filename=true, hive_partitioning=1)
# WHERE addresses[1].freeform IS NOT NULL
# AND
#     bbox.xmin BETWEEN -73.73 AND -71.78 AND
#     bbox.ymin BETWEEN 40.98 AND 42.06 AND
#     confidence > 0.5 AND
#     UPPER(addresses[1].region) = 'CT'
# LIMIT 5
# """).df()

# con.sql("""
# CREATE TABLE ct_places AS
# SELECT id, names.primary AS name, UPPER(addresses[1].freeform) AS freeform, UPPER(addresses[1].locality) AS locality, addresses[1].postcode AS postcode, addresses[1].region AS region, confidence, sources, bbox
# FROM read_parquet('s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/*.zstd.parquet')
# WHERE addresses[1].freeform IS NOT NULL
# AND
#     bbox.xmin BETWEEN -73.73 AND -71.78 AND
#     bbox.ymin BETWEEN 40.98 AND 42.06 AND
#     (UPPER(addresses[1].region) = 'CT'
#     OR (addresses[1].region IS NULL AND addresses[1].postcode LIKE '06%'))
# """)

# con.sql("""
#     COPY ct_places TO
#     'Overture/ct_places.parquet' (FORMAT parquet)
# """)

# con.sql("""
#     SELECT COUNT(*) FROM 
#     read_parquet('Overture/ct_places.parquet')
# """)

┌──────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │                                                                                                                         column_type                                                                                                                         │  null   │   key   │ default │  extra  │
│     varchar      │                                                                                                                           varchar                                                                                                                           │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼───────────────────────────────────

In [ ]:
# con.sql("""
#     SELECT * FROM
#     read_parquet('Overture/ct_places_2026-09-23.1.parquet')
#     LIMIT 5
# """).df()

# con.sql("""
#     DESCRIBE SELECT * FROM 
#     read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/part-00000-10b8d380-b245-47f7-9ce1-f0de1f8903e6.c000.snappy.parquet')
# """).df()

# con.sql("""
#     SELECT sources[1].property AS property FROM 
#     read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/part-00000-10b8d380-b245-47f7-9ce1-f0de1f8903e6.c000.snappy.parquet')
#     LIMIT 5
# """).df()

con.sql("""
    SELECT LOWER(p.sources[1].dataset) AS published_from, COUNT(DISTINCT(p.id)) AS N
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/*.parquet') AS q
    ON p.id = q.id
    GROUP BY published_from
""")

# con.sql("""
#     SELECT *
#     FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
#     JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/*.parquet') AS q
#     ON p.id = q.id
# """)

# con.sql("""
#     SELECT LOWER(p.sources[1].dataset) as published_from, COUNT(distinct(p.id)) as N
#     FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
#     JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=alltheplaces/theme=places/type=place/*.parquet') AS q
#     ON p.id = q.id
#     GROUP BY published_from
# """)

# con.sql("""
#     SELECT record_id
#     FROM read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=alltheplaces/theme=places/type=place/part-00000-10b8d380-b245-47f7-9ce1-f0de1f8903e6.c000.snappy.parquet')
#     LIMIT 5
# """)

con.sql("""
    CREATE TABLE ct_bridge AS 

    SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/*.parquet') AS q
    ON p.id = q.id

    UNION ALL

    SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=alltheplaces/theme=places/type=place/*.parquet') AS q
    ON p.id = q.id
""")

┌────────────────┬───────┐
│ published_from │   N   │
│    varchar     │ int64 │
├────────────────┼───────┤
│ meta           │  4867 │
│ microsoft      │  1041 │
│ brightquery    │   319 │
│ dac            │   219 │
│ alltheplaces   │  2067 │
└────────────────┴───────┘